In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

# --- 1. Define Paths and Parameters ---
# This points to the dataset you just added
base_dir = '../data/FER2013_images/train'
# We only care about these two classes for our specialized model
CLASSES_TO_USE = ['sad', 'neutral']
IMAGE_SIZE = (48, 48)
BATCH_SIZE = 32

# --- 2. Load and Prepare the Data ---
# We use ImageDataGenerator to load images directly from the folders
datagen = ImageDataGenerator(
    rescale=1./255,         # Normalize pixel values to be between 0 and 1
    validation_split=0.2    # Automatically use 20% of the data for testing
)

# Create a generator for the training data
train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',     # For two classes (sad vs. neutral)
    color_mode='grayscale',  # The images are grayscale
    classes=CLASSES_TO_USE,
    subset='training'
)

# Create a generator for the validation (testing) data
validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale',
    classes=CLASSES_TO_USE,
    subset='validation'
)

# --- 3. Build the Fine-Tuned Model ---
# Load the powerful MobileNetV2 model, pre-trained on millions of images
# We exclude the final layer (include_top=False) because we will replace it
base_model = MobileNetV2(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 1), # Grayscale, so 1 channel
    alpha=1.0,
    include_top=False,
    weights=None # Start with fresh weights for this specific grayscale task
)

# Freeze the early layers of the model
# for layer in base_model.layers:
#     layer.trainable = False

# Add our custom final layers for our specific task
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
# The final output layer has one neuron with a sigmoid activation,
# which will output a probability between 0 (neutral) and 1 (sad)
predictions = Dense(1, activation='sigmoid')(x)

# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nModel Summary:")
model.summary()

# --- 4. Train the Model ---
print("\nStarting to fine-tune the emotion model...")
history = model.fit(
    train_generator,
    epochs=10, # Train for 10 full passes through the data
    validation_data=validation_generator
)
print("Model fine-tuning complete.")

# --- 5. Save the New, More Sensitive Model ---
# We save it in the Keras format. The next step will be to convert it.
model.save('../models/emotion_model_sad_vs_neutral.h5')
print("\n✅ New, more sensitive model saved to models/emotion_model_sad_vs_neutral.h5")

Found 7836 images belonging to 2 classes.
Found 1959 images belonging to 2 classes.

Model Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 48, 48, 1)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 24, 24, 32)        │             288 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 24, 24, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 24, 24, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 24, 24, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 24, 24, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 24, 24, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 24, 24, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 24, 24, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 24, 24, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 24, 24, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 24, 24, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 25, 25, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 12, 12, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,421,505 (9.24 MB)

 Trainable params: 2,387,393 (9.11 MB)

 Non-trainable params: 34,112 (133.25 KB)


Starting to fine-tune the emotion model...


C:\Users\91952\Desktop\FINAL_PROJECT-1\mental-health-detection\backend\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 296s 932ms/step - accuracy: 0.5420 - loss: 0.7027 - val_accuracy: 0.4931 - val_loss: 0.6933
Epoch 2/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 64s 259ms/step - accuracy: 0.6110 - loss: 0.6721 - val_accuracy: 0.4931 - val_loss: 0.6939
Epoch 3/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 40s 162ms/step - accuracy: 0.6279 - loss: 0.6535 - val_accuracy: 0.4931 - val_loss: 0.6950
Epoch 4/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 36s 148ms/step - accuracy: 0.6515 - loss: 0.6340 - val_accuracy: 0.4931 - val_loss: 0.6971
Epoch 5/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 37s 149ms/step - accuracy: 0.6727 - loss: 0.6200 - val_accuracy: 0.4931 - val_loss: 0.7011
Epoch 6/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 36s 149ms/step - accuracy: 0.6759 - loss: 0.6100 - val_accuracy: 0.4931 - val_loss: 0.7063
Epoch 7/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 37s 150ms/step - accuracy: 0.6919 - loss: 0.5903 - val_accuracy: 0.4931 - val_loss: 0.7092
Epoch 8/10
245/245 ━━━━━━━━━━━━━━━━━━━━ 38s 153ms/step - accuracy: 0.7134 - loss: 

Model fine-tuning complete.

✅ New, more sensitive model saved to models/emotion_model_sad_vs_neutral.h5
